# Assignment 5

In [13]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter('ignore')

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

import statsmodels.formula.api as smf

# --- Intentamos importar hdmpy y definir RLasso real ---
try:
    import hdmpy
    from sklearn.base import BaseEstimator

    class RLasso(BaseEstimator):
        def __init__(self, *, post=True):
            self.post = post

        def fit(self, X, y):
            self.rlasso_ = hdmpy.rlasso(X, y, post=self.post)
            return self

        def predict(self, X):
            return (
                np.array(X) @ np.array(self.rlasso_.est['beta']).flatten()
                + np.array(self.rlasso_.est['intercept'])
            )

    have_rlasso = True

# --- Si hdmpy no está disponible, usamos LassoCV como fallback ---
except Exception as e:
    from sklearn.base import BaseEstimator

    class RLasso(BaseEstimator):
        def __init__(self, *, post=True):
            self.post = post
            self.model = LassoCV(cv=5)

        def fit(self, X, y):
            self.model.fit(X, y)
            return self

        def predict(self, X):
            return self.model.predict(X)

    have_rlasso = False
    print("hdmpy.RLasso no disponible; utilizando LassoCV como fallback para RLasso.")

# Fijamos semilla
np.random.seed(123)


hdmpy.RLasso no disponible; utilizando LassoCV como fallback para RLasso.


## I. Cleaning and set-up

In [14]:
# ---------------------------
# I. Cleaning and set-up
# ---------------------------

file_path = "C:/Users/VICTOR/Documents/GitHub/UP_courses/penn_jae.dat.txt"   # AJUSTA ruta si hace falta

# Leer archivo .dat (asume whitespace / tab delimited). Si es CSV, cambiar read_table->read_csv.
if not os.path.exists(file_path):
    raise FileNotFoundError(f"No encontré el archivo '{file_path}' en el directorio actual. Ajusta file_path.")

data = pd.read_table(file_path, header=0, sep=None, engine='python')  # intenta detectar separador
# Quita columnas no nombradas si aparecen
data = data.loc[:, ~data.columns.str.contains('^Unnamed')]

# 1. Mantener solo tg == 0 o tg == 4
data = data[data['tg'].isin([0, 4])].copy()

# 2. T4 dummy
data['T4'] = (data['tg'] == 4).astype(int)

# 3. outcome y = log(inuidur1)
# Manejar ceros/NA en inuidur1: eliminar o imputar. Aquí quitamos observaciones con inuidur1 <= 0 o NA.
data = data[~data['inuidur1'].isna()]
data = data[data['inuidur1'] > 0].copy()
data['y'] = np.log(data['inuidur1'])

# 4. dummies para dep (dep_0, dep_1, dep_2)
dep_dummies = pd.get_dummies(data['dep'].astype(int), prefix='dep')
# Asegurar que existen las 3 columnas (si falta alguna, crearla con 0)
for col in ['dep_0', 'dep_1', 'dep_2']:
    if col not in dep_dummies.columns:
        dep_dummies[col] = 0
# join
data = pd.concat([data, dep_dummies[['dep_0', 'dep_1', 'dep_2']]], axis=1)

# 5. construir X con las variables pedidas
x_vars = [
    'female','black','othrace',
    'dep_1','dep_2',
    'q2','q3','q4','q5','q6',
    'recall','agelt35','agegt54',
    'durable','nondurable','lusd','husd'
]

# Ver que todas las variables existan en el dataset
missing = [v for v in x_vars if v not in data.columns]
if len(missing) > 0:
    raise ValueError(f"Faltan columnas esperadas en el dataset: {missing}")

X = data[x_vars].copy()
y = data['y'].values
d = data['T4'].values

print("Datos preparados:")
print("N obs:", data.shape[0])
print("N covariables x:", X.shape[1])

Datos preparados:
N obs: 5099
N covariables x: 17


## II. Debiased ML

In [16]:
# ---------------------------
# II. Debiased ML (with cross-fitting)
# ---------------------------

def dml(X_df, D, y, modely, modeld, *, nfolds=5, classifier=False, cluster=False):
    """
    DML con cross-fitting (estilo de los notebooks de la clase).
    Retorna: point, stderr, yhat, Dhat, resy, resD, epsilon
    """
    X_np = np.array(X_df)
    cv = KFold(n_splits=nfolds, shuffle=True, random_state=123)
    # out-of-fold predictions
    yhat = cross_val_predict(modely, X_np, y, cv=cv, n_jobs=-1)
    if classifier:
        # predict_proba -> second column
        Dhat_probs = cross_val_predict(modeld, X_np, D, cv=cv, method='predict_proba', n_jobs=-1)
        # If predict_proba returns Nx2, take column 1
        if Dhat_probs.ndim == 2:
            Dhat = Dhat_probs[:,1]
        else:
            Dhat = Dhat_probs
    else:
        Dhat = cross_val_predict(modeld, X_np, D, cv=cv, n_jobs=-1)

    resy = y - yhat
    resD = D - Dhat

    dml_data = pd.DataFrame({'resy': resy, 'resD': resD})
    # final OLS residual-on-residual
    ols_mod = smf.ols(formula='resy ~ 1 + resD', data=dml_data).fit()
    point = ols_mod.params['resD']
    stderr = ols_mod.bse['resD']
    epsilon = ols_mod.resid

    return point, stderr, yhat, Dhat, resy, resD, epsilon

def summary_df(point, stderr, yhat, Dhat, resy, resD, epsilon, X_df, D, y, name):
    return pd.DataFrame({
        'estimate': [point],
        'stderr': [stderr],
        'rmse y': [np.sqrt(np.mean(resy**2))],
        'rmse D': [np.sqrt(np.mean(resD**2))]
    }, index=[name])

# Preparar modelos (pipelines)
modely_OLS = make_pipeline(StandardScaler(), LinearRegression())
modeld_OLS = make_pipeline(StandardScaler(), LinearRegression())

modely_RLasso = make_pipeline(StandardScaler(), RLasso(post=False))
modeld_RLasso = make_pipeline(StandardScaler(), RLasso(post=False))

modely_RF = make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=123))
modeld_RF = make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=123))

# Neural Network (MLPRegressor)
modely_NN = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(50,25), max_iter=2000, random_state=123))
modeld_NN = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(50,25), max_iter=2000, random_state=123))

# Ejecutar DML (cross-fitting) para varios modelos
results_tables = []

# OLS
res_OLS = dml(X, d, y, modely_OLS, modeld_OLS, nfolds=10, classifier=False)
table_OLS = summary_df(*res_OLS, X, d, y, name='DML_OLS')
results_tables.append(table_OLS)

# RLasso (o LassoCV fallback)
res_RLasso = dml(X, d, y, modely_RLasso, modeld_RLasso, nfolds=10, classifier=False)
table_RLasso = summary_df(*res_RLasso, X, d, y, name='DML_RLasso')
results_tables.append(table_RLasso)

# Random Forest
res_RF = dml(X, d, y, modely_RF, modeld_RF, nfolds=10, classifier=False)
table_RF = summary_df(*res_RF, X, d, y, name='DML_RF')
results_tables.append(table_RF)

# Neural Network (mix: NN for y, RF for d)
res_NN = dml(X, d, y, modely_NN, modeld_RF, nfolds=10, classifier=False)
table_NN = summary_df(*res_NN, X, d, y, name='DML_NN/RF')
results_tables.append(table_NN)

# DML with Neural Network for both y and d (NN/NN)
modely = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(50, 20),
                 activation='relu',
                 max_iter=1000,
                 random_state=123)
)
modeld = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(50, 20),
                 activation='relu',
                 max_iter=1000,
                 random_state=123)
)

# ✅ Cambié x → X (mayúscula)
result_NN_NN = dml(X, d, y, modely, modeld, nfolds=10, classifier=False, cluster=False)

# ✅ Crear tabla y añadirla a los resultados
table_NN_NN = summary_df(*result_NN_NN, X, d, y, name='DML_NN/NN')
results_tables.append(table_NN_NN)

# ✅ Cambié table_mix → table_NN
table = pd.concat([table_OLS, table_RLasso, table_RF, table_NN, table_NN_NN], axis=0)

table_dml = pd.concat(results_tables)
print("\n=== Results: DML with cross-fitting ===")
print(table_dml)

# Selección de modelo: imprimir RMSEs y estimates para comparar
print("\nComparación rápida (DML):")
print(table_dml.sort_values('estimate'))




=== Results: DML with cross-fitting ===
            estimate    stderr    rmse y    rmse D
DML_OLS    -0.070013  0.035190  1.194292  0.475189
DML_RLasso -0.070349  0.035242  1.194500  0.474566
DML_RF     -0.076879  0.035340  1.217526  0.482339
DML_NN/RF  -0.081204  0.036677  1.263723  0.482339
DML_NN/NN  -0.059286  0.034980  1.260565  0.504492

Comparación rápida (DML):
            estimate    stderr    rmse y    rmse D
DML_NN/RF  -0.081204  0.036677  1.263723  0.482339
DML_RF     -0.076879  0.035340  1.217526  0.482339
DML_RLasso -0.070349  0.035242  1.194500  0.474566
DML_OLS    -0.070013  0.035190  1.194292  0.475189
DML_NN/NN  -0.059286  0.034980  1.260565  0.504492


## III. No cross-fitting

In [10]:
# ---------------------------
# III. No cross-fitting (in-sample predictions)
# ---------------------------

def dml_no_cf(X_df, D, y, modely, modeld, *, classifier=False):
    """
    DML sin cross-fitting: ajusta modelos sobre todo el conjunto y utiliza las predicciones in-sample.
    Retorna los mismos objetos que dml().
    """
    X_np = np.array(X_df)
    # entrenar en todo el conjunto
    modely_fit = clone_model(modely)
    modeld_fit = clone_model(modeld)
    modely_fit.fit(X_np, y)
    if classifier:
        modeld_fit.fit(X_np, D)
        # predict_proba
        try:
            Dhat = modeld_fit.predict_proba(X_np)[:,1]
        except:
            Dhat = modeld_fit.predict(X_np)
    else:
        modeld_fit.fit(X_np, D)
        Dhat = modeld_fit.predict(X_np)
    yhat = modely_fit.predict(X_np)

    resy = y - yhat
    resD = D - Dhat

    dml_data = pd.DataFrame({'resy': resy, 'resD': resD})
    ols_mod = smf.ols(formula='resy ~ 1 + resD', data=dml_data).fit()
    point = ols_mod.params['resD']
    stderr = ols_mod.bse['resD']
    epsilon = ols_mod.resid

    return point, stderr, yhat, Dhat, resy, resD, epsilon

# helper: clone estimator (simple wrapper)
def clone_model(pipe):
    from sklearn.base import clone
    return clone(pipe)

# Ejecutar no cross-fitting
results_no_cf = []

res_OLS_nc = dml_no_cf(X, d, y, modely_OLS, modeld_OLS, classifier=False)
results_no_cf.append(summary_df(*res_OLS_nc, X, d, y, name='NoCF_OLS'))

res_RLasso_nc = dml_no_cf(X, d, y, modely_RLasso, modeld_RLasso, classifier=False)
results_no_cf.append(summary_df(*res_RLasso_nc, X, d, y, name='NoCF_RLasso'))

res_RF_nc = dml_no_cf(X, d, y, modely_RF, modeld_RF, classifier=False)
results_no_cf.append(summary_df(*res_RF_nc, X, d, y, name='NoCF_RF'))

res_NN_nc = dml_no_cf(X, d, y, modely_NN, modeld_RF, classifier=False)
results_no_cf.append(summary_df(*res_NN_nc, X, d, y, name='NoCF_NN/RF'))

table_no_cf = pd.concat(results_no_cf)
print("\n=== Results: DML without cross-fitting (in-sample preds) ===")
print(table_no_cf)

# ---------------------------
# Comparación y respuestas
# ---------------------------
print("\n=== Comparación RMSEs entre DML (cross-fit) y NoCF ===")
compare = pd.concat([table_dml.add_prefix('CF_'), table_no_cf.add_prefix('NoCF_')], axis=1)
print(compare)


=== Results: DML without cross-fitting (in-sample preds) ===
             estimate    stderr    rmse y    rmse D
NoCF_OLS    -0.071216  0.035186  1.189571  0.473352
NoCF_RLasso -0.071227  0.035151  1.189576  0.473824
NoCF_RF     -0.074878  0.035200  1.125760  0.447765
NoCF_NN/RF  -0.072088  0.033942  1.086238  0.447765

=== Comparación RMSEs entre DML (cross-fit) y NoCF ===
             CF_estimate  CF_stderr  CF_rmse y  CF_rmse D  NoCF_estimate  \
DML_OLS        -0.070013   0.035190   1.194292   0.475189            NaN   
DML_RLasso     -0.070349   0.035242   1.194500   0.474566            NaN   
DML_RF         -0.076879   0.035340   1.217526   0.482339            NaN   
DML_NN/RF      -0.081204   0.036677   1.263723   0.482339            NaN   
NoCF_OLS             NaN        NaN        NaN        NaN      -0.071216   
NoCF_RLasso          NaN        NaN        NaN        NaN      -0.071227   
NoCF_RF              NaN        NaN        NaN        NaN      -0.074878   
NoCF_NN/RF    